# `%provenance` magic demo

Ask for citations from inside the notebook you are working in, instead of calling the workflow functions directly.

Run this in the `lang` conda env. The cells below are not pre-run: `%provenance` calls Gemini to route the request, so the output depends on your `GOOGLE_API_KEY` in `src/.env`.

## Load the extension

The repo has no packaging, so `src/` goes on `sys.path` first - the same convention `workflow.ipynb` and the tests use.

In [ ]:
import sys
sys.path.append("../src")

%load_ext provenance

## Point it at a notebook

`%provenance` auto-detects the running notebook via `ipynbname`. That works in classic Jupyter but usually fails in VSCode, because it matches the kernel against the Jupyter server's session list. When it fails you get a one-line `UsageError` telling you to run this:

In [ ]:
%provenance_notebook testing/paleoPCAlite.ipynb

## Cite the software

Routes to `cite_software`, which extracts imports with AST and matches them against `Citations/`. Output defaults to APA.

In [ ]:
%provenance cite the software

### One specific library

The agent passes the library name through as a filter.

In [ ]:
%provenance cite Pyleoclim

### Raw BibTeX

Say so in the request - there is no separate flag. This path also skips the LLM entirely, so it is the fast one.

In [ ]:
%provenance cite the software in BibTeX

## Cite the datasets

Routes to `cite_data`, which detects dataset variables with the LLM and **injects a retrieval cell per dataset into the notebook file**.

The citations are the output of those injected cells, not of this one - retrieval needs the live objects already loaded in that notebook's kernel. So the magic reports what it wrote, and you reload the target notebook and run the new cells.

Note it writes **in place**. Reload the target notebook before editing it further, or an editor save can overwrite the injected cells.

In [ ]:
%provenance cite the datasets

## Notes

- Every workflow reads the `.ipynb` **from disk**. A cell you typed but have not saved is invisible, so save before asking.
- An unroutable request (`%provenance what is the weather`) prints what the two tools cover rather than failing silently.
- APA rendering currently makes one Gemini call per entry. Issue #29 replaces it with a deterministic formatter, after which the software path is fully offline.